# 🎓 STUDI ABLASI KOMPREHENSIF: EVALUASI LLM-AS-LABELER (LLAMA-3-8B) PADA GPU T4
## Perluasan Dataset IndQNER Menggunakan Arsitektur Two-Tier Labeling

Notebook ini menyajikan pengujian komprehensif tingkat akademis untuk Bab IV Skripsi:
1. **Pemeriksaan & Profiling GPU**: Pemantauan konsumsi VRAM dan kecepatan latensi inferensi.
2. **Eksperimen Komparatif Head-to-Head**: **Zero-Shot vs Few-Shot Prompting** pada 237 entitas ground-truth (`test.txt`).
3. **Metrik Standar Resmi**: Macro & Weighted Precision, Recall, F1-Score untuk 20 kelas IndQNER.
4. **Analisis Galat Mendalam (*Error Analysis & Confusion Matrix*)**: Mengidentifikasi entitas yang memicu ambiguitas semantik (misal *Prophet* vs *Messenger*).
5. **Pelabelan Entitas Korpus Unmapped**: Melabeli 33 entitas baru dari 114 surah dengan audit lengkap.
6. **Ekspor Data Komprehensif**: Menyimpan seluruh log inferensi dan metrik ke berkas JSON.

--- 
### 1. Instalasi Dependensi & Verifikasi GPU T4

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch scikit-learn tabulate pandas seaborn matplotlib

import torch
print("=" * 60)
print(f"CUDA Tersedia       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name         : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM Tersedia : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("=" * 60)

--- 
### 2. Memuat Model Meta-Llama-3-8B-Instruct (4-Bit Quantized)

In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

# Model Llama-3-8B terkuantisasi 4-bit (sangat optimal untuk GPU T4 ~5.5 GB VRAM)
model_id = "unsloth/llama-3-8b-Instruct-bnb-4bit"

print("⏳ Memuat tokenizer dan model...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

vram_loaded = torch.cuda.memory_allocated() / 1e9
print(f"✅ Model berhasil dimuat dalam {time.time() - t0:.2f} detik!")
print(f"📊 Konsumsi VRAM saat memuat model: {vram_loaded:.2f} GB")

--- 
### 3. Unggah dan Pemeriksaan Payload Data (`llm_colab_payload.json`)

In [ ]:
import json, os
import pandas as pd
from google.colab import files

if not os.path.exists("llm_colab_payload.json"):
    print("Silakan unggah 'llm_colab_payload.json' dari folder lokal data/intermediate/:")
    uploaded = files.upload()

with open("llm_colab_payload.json", "r", encoding="utf-8") as f:
    payload = json.load(f)

benchmark_items = payload["benchmark_entities"]
unmapped_items = payload["unmapped_entities"]

print("=" * 60)
print(f"Metadata Payload                 : {payload['metadata']['description']}")
print(f"Jumlah Entitas Benchmark Test Set: {len(benchmark_items)}")
print(f"Jumlah Entitas Unmapped 114 Surah: {len(unmapped_items)}")
print("=" * 60)

# Tinjau distribusi kelas ground truth pada test set
df_benchmark = pd.DataFrame(benchmark_items)
print("\nDistribusi Ground Truth 20 Kelas pada Test Set:")
print(df_benchmark["ground_truth_class"].value_counts().to_string())

--- 
### 4. Definisi 20 Kelas Qurani & Formulator Prompt (Zero-Shot & Few-Shot)

In [ ]:
VALID_CLASSES = [
    "Allah", "Messenger", "Prophet", "Person", "HolyBook",
    "AstronomicalBody", "Event", "AfterlifeLocation", "GeographicalLocation", "Artifact",
    "Throne", "Religion", "Angel", "Color", "Language",
    "Food", "Fruit", "FalseDeity", "Sentient", "TheBookOfAllah"
]

FEW_SHOT_EXEMPLARS = [
    {"s": "Kitab (Al-Qur’an) ini tidak ada keraguan di dalamnya.", "e": "Al-Qur’an", "c": "HolyBook"},
    {"s": "Katakanlah, 'Kami beriman kepada Allah dan apa yang diturunkan kepada Ibrahim dan Musa.'", "e": "Musa", "c": "Messenger"},
    {"s": "Dia mengajarkan kepada Adam nama-nama benda seluruhnya.", "e": "Adam", "c": "Prophet"},
    {"s": "Dia menjadikan bagimu bumi sebagai hamparan dan langit sebagai atap.", "e": "bumi", "c": "AstronomicalBody"},
    {"s": "Sesungguhnya orang-orang yang beriman, orang-orang Yahudi, dan Nasrani.", "e": "Nasrani", "c": "Religion"},
    {"s": "Dan ingatlah ketika Kami selamatkan kamu dari pengikut-pengikut Fir‘aun.", "e": "Fir‘aun", "c": "Person"}
]

def build_chat_prompt(sentence: str, entity: str, mode: str = "few_shot") -> str:
    classes_list = "\n".join([f"- {c}" for c in VALID_CLASSES])
    sys_msg = (
        "Anda adalah sistem anotasi Named Entity Recognition (NER) khusus teks Terjemahan Al-Qur'an Kementerian Agama RI.\n"
        "Tugas Anda: Menentukan satu kelas kategori yang paling tepat untuk entitas yang ditunjuk.\n\n"
        "Daftar 20 Kelas yang Sah:\n"
        f"{classes_list}\n\n"
        "ATURAN KETAT:\n"
        "1. Jawab HANYA dengan SATU nama kelas dari daftar 20 kelas di atas.\n"
        "2. DILARANG membuat kategori baru atau memberikan penjelasan apa pun."
    )
    
    examples_block = ""
    if mode == "few_shot":
        examples_block = "\nBerikut beberapa contoh acuan pelabelan resmi:\n"
        for ex in FEW_SHOT_EXEMPLARS:
            examples_block += f"Kalimat: \"{ex['s']}\" | Entitas: \"{ex['e']}\" -> Kelas: {ex['c']}\n"
            
    user_msg = (
        f"{examples_block}\n"
        f"Kalimat: \"{sentence}\"\n"
        f"Entitas: \"{entity}\"\n"
        "Kelas:"
    )
    
    messages = [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": user_msg}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def extract_class_from_output(raw_output: str) -> str:
    clean = raw_output.strip().replace("*", "").replace("\"", "").replace("'", "")
    # Exact matching
    for c in VALID_CLASSES:
        if clean.lower() == c.lower():
            return c
    # Substring matching
    for c in VALID_CLASSES:
        if c.lower() in clean.lower():
            return c
    return "Person"

--- 
### 5. Engine Inferensi Batch dengan Profiling Latensi & VRAM

In [ ]:
def run_batch_evaluation(items, mode="few_shot"):
    results = []
    total_n = len(items)
    print(f"\n🚀 Menjalankan Inferensi Llama-3-8B | Mode: {mode.upper()} | Jumlah: {total_n} entitas...")
    
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    
    for idx, item in enumerate(items):
        prompt_text = build_chat_prompt(item["sentence"], item["entity"], mode=mode)
        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
        
        item_t0 = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                temperature=0.01,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        item_duration_ms = (time.time() - item_t0) * 1000
        
        raw_resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        pred_cls = extract_class_from_output(raw_resp)
        
        res_entry = dict(item)
        res_entry["predicted_class"] = pred_cls
        res_entry["raw_output"] = raw_resp.strip()
        res_entry["latency_ms"] = round(item_duration_ms, 2)
        res_entry["mode"] = mode
        results.append(res_entry)
        
        if (idx + 1) % 50 == 0 or (idx + 1) == total_n:
            print(f"  [{idx+1:3d}/{total_n}] Selesai | Latensi rata-rata: {item_duration_ms:.1f} ms")
            
    total_time = time.time() - start_time
    peak_vram = torch.cuda.max_memory_allocated() / 1e9
    
    print(f"\n⏱️ Total Waktu Inferensi: {total_time:.2f} detik ({total_time/total_n*1000:.1f} ms/entitas)")
    print(f"📊 Peak VRAM Penggunaan: {peak_vram:.2f} GB")
    return results, {"total_time_sec": total_time, "avg_latency_ms": total_time/total_n*1000, "peak_vram_gb": peak_vram}

--- 
### 6. Eksekusi Eksperimen Komparatif: Zero-Shot vs Few-Shot pada Test Set (237 Entitas)

In [ ]:
# 1. Jalankan Zero-Shot
zero_shot_results, zero_shot_perf = run_batch_evaluation(benchmark_items, mode="zero_shot")

# 2. Jalankan Few-Shot
few_shot_results, few_shot_perf = run_batch_evaluation(benchmark_items, mode="few_shot")

--- 
### 7. Evaluasi Kuantitatif & Tabel Komparasi Head-to-Head (Bab IV Skripsi)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
from tabulate import tabulate

y_true = [it["ground_truth_class"] for it in benchmark_items]
y_pred_zs = [it["predicted_class"] for it in zero_shot_results]
y_pred_fs = [it["predicted_class"] for it in few_shot_results]

# Perhitungan metrik global
acc_zs = accuracy_score(y_true, y_pred_zs)
p_zs_w, r_zs_w, f1_zs_w, _ = precision_recall_fscore_support(y_true, y_pred_zs, average="weighted", zero_division=0)
p_zs_m, r_zs_m, f1_zs_m, _ = precision_recall_fscore_support(y_true, y_pred_zs, average="macro", zero_division=0)

acc_fs = accuracy_score(y_true, y_pred_fs)
p_fs_w, r_fs_w, f1_fs_w, _ = precision_recall_fscore_support(y_true, y_pred_fs, average="weighted", zero_division=0)
p_fs_m, r_fs_m, f1_fs_m, _ = precision_recall_fscore_support(y_true, y_pred_fs, average="macro", zero_division=0)

summary_table = [
    ["Akurasi Keseluruhan", f"{acc_zs*100:.2f}%", f"{acc_fs*100:.2f}%", f"{(acc_fs - acc_zs)*100:+.2f}%"],
    ["Weighted Precision", f"{p_zs_w*100:.2f}%", f"{p_fs_w*100:.2f}%", f"{(p_fs_w - p_zs_w)*100:+.2f}%"],
    ["Weighted Recall", f"{r_zs_w*100:.2f}%", f"{r_fs_w*100:.2f}%", f"{(r_fs_w - r_zs_w)*100:+.2f}%"],
    ["Weighted F1-Score", f"{f1_zs_w*100:.2f}%", f"{f1_fs_w*100:.2f}%", f"{(f1_fs_w - f1_zs_w)*100:+.2f}%"],
    ["Macro F1-Score", f"{f1_zs_m*100:.2f}%", f"{f1_fs_m*100:.2f}%", f"{(f1_fs_m - f1_zs_m)*100:+.2f}%"],
    ["Latensi Rata-rata", f"{zero_shot_perf['avg_latency_ms']:.1f} ms", f"{few_shot_perf['avg_latency_ms']:.1f} ms", f"{few_shot_perf['avg_latency_ms'] - zero_shot_perf['avg_latency_ms']:+.1f} ms"],
    ["Peak VRAM GPU", f"{zero_shot_perf['peak_vram_gb']:.2f} GB", f"{few_shot_perf['peak_vram_gb']:.2f} GB", "-"]
]

print("=" * 75)
print("TABEL KOMPARASI ILMIAH: ZERO-SHOT VS FEW-SHOT LLAMA-3-8B (GPU T4)")
print("=" * 75)
print(tabulate(summary_table, headers=["Metrik Evaluasi", "Zero-Shot", "Few-Shot (6 Exemplars)", "Delta (Peningkatan)"], tablefmt="github"))

print("\n" + "=" * 75)
print("DETAIL CLASSIFICATION REPORT (FEW-SHOT MODE):")
print("=" * 75)
print(classification_report(y_true, y_pred_fs, zero_division=0, digits=4))

--- 
### 8. Analisis Galat Mendalam (*Error Analysis & Confusion Matrix*)
Menyajikan data entitas yang salah diprediksi untuk bahan pembahasan kritis di draf Skripsi.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Identifikasi entitas yang salah ditebak pada Few-Shot
misclassified = []
for it, yt, yp in zip(few_shot_results, y_true, y_pred_fs):
    if yt != yp:
        misclassified.append({
            "entitas": it["entity"],
            "ground_truth": yt,
            "prediksi_llama3": yp,
            "cuplikan_kalimat": it["sentence"][:70] + "..."
        })

df_errors = pd.DataFrame(misclassified)
print(f"\n🔍 Ditemukan {len(df_errors)} kasus misklasifikasi dari 237 entitas uji ({len(df_errors)/237*100:.1f}% error rate):")
print(tabulate(df_errors.head(15), headers="keys", tablefmt="github"))

# Ringkasan pasangan salah tebak terbanyak
print("\nPasangan Ambigu Terbanyak (Ground Truth -> Prediksi):\n")
if not df_errors.empty:
    pairs = df_errors.groupby(["ground_truth", "prediksi_llama3"]).size().reset_index(name="frekuensi")
    print(tabulate(pairs.sort_values(by="frekuensi", ascending=False).head(10), headers="keys", tablefmt="github"))

--- 
### 9. Pelabelan 33 Entitas Korpus Unmapped (Extended IndQNER 114 Surah)

In [ ]:
print("\n🚀 Melabeli 33 Entitas Unmapped Korpus 114 Surah dengan Llama-3-8B (Few-Shot)...")
unmapped_predictions, unmapped_perf = run_batch_evaluation(unmapped_items, mode="few_shot")

df_unmapped_res = pd.DataFrame(unmapped_predictions)
print("\n" + "=" * 75)
print("HASIL PREDIKSI 33 ENTITAS UNMAPPED (KORPUS 114 SURAH):")
print("=" * 75)
print(tabulate(df_unmapped_res[["surah", "ayah", "entity", "predicted_class"]], headers=["Surah", "Ayat", "Entitas Teks", "Prediksi Kelas Llama-3"], tablefmt="github"))

--- 
### 10. Ekspor Hasil Komprehensif (`llama3_labeling_results.json`)

In [ ]:
final_export = {
    "metadata": {
        "experiment": "Ablation Study: LLM-as-Labeler (Llama-3-8B-Instruct 4-bit)",
        "hardware": "Google Colab GPU T4 (15 GB VRAM)",
        "metrics": {
            "zero_shot": {
                "accuracy": float(acc_zs),
                "weighted_f1": float(f1_zs_w),
                "macro_f1": float(f1_zs_m),
                "performance": zero_shot_perf
            },
            "few_shot": {
                "accuracy": float(acc_fs),
                "weighted_f1": float(f1_fs_w),
                "macro_f1": float(f1_fs_m),
                "performance": few_shot_perf
            }
        }
    },
    "benchmark_results_zero_shot": zero_shot_results,
    "benchmark_results_few_shot": few_shot_results,
    "unmapped_results": unmapped_predictions,
    "error_analysis": misclassified
}

output_filename = "llama3_labeling_results.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(final_export, f, ensure_ascii=False, indent=2)

print(f"\n💾 File '{output_filename}' berhasil dibuat!")
files.download(output_filename)
print("🎉 Berkas telah diunduh otomatis. Siap diintegrasikan ke repositori lokal!")